# A County-Level Suitability Analysis for New Data Center Sites in the United States

In [ ]:
import arcpy
import os
import pandas as pd
import math

# arcpy.env.parallelProcessingFactor = "0" 
# arcpy.env.scratchWorkspace = "memory"
conus_albers_sr = arcpy.SpatialReference(5070)
print(arcpy.CheckOutExtension("Spatial"))
arcpy.env.overwriteOutput = True

from arcpy.sa import *

data_dir = r"C:\.Jyothir\GEOG 4480\suitable-data-center\data"
# data_dir = r"D:\Projects\suitable-data-center\data"
# data_dir = r"C:\Users\jharde03\Documents\work\data"

# temp_gdb = os.path.join(data_dir, "temp.gdb")
# if not arcpy.Exists(temp_gdb):
#     arcpy.management.CreateFileGDB(data_dir, "temp.gdb")

arcpy.env.workspace = data_dir
# arcpy.env.scratchWorkspace = temp_gdb

CheckedOut


In [ ]:
counties_shp = os.path.join(data_dir, "US_Contig_County_Boundaries", "US_County_Boundaries.shp")
copy_dir = os.path.join(data_dir, "Counties_Suitability_Master")
unproj_counties = os.path.join(copy_dir, "Counties_Suitability_Unprojected.shp")
master_counties = os.path.join(copy_dir, "Counties_Suitability_Master.shp")
os.makedirs(copy_dir, exist_ok=True)

arcpy.management.CopyFeatures(counties_shp, unproj_counties)

arcpy.management.Project(
    in_dataset=unproj_counties,
    out_dataset=master_counties,
    out_coor_system=conus_albers_sr
)

state_fips = {
    'Alabama': '01', 'Arizona': '04', 'Arkansas': '05', 'California': '06',
    'Colorado': '08', 'Connecticut': '09', 'Delaware': '10', 'District of Columbia': '11', 
    'Florida': '12', 'Georgia': '13', 'Idaho': '16', 'Illinois': '17', 'Indiana': '18',
    'Iowa': '19', 'Kansas': '20', 'Kentucky': '21', 'Louisiana': '22',
    'Maine': '23', 'Maryland': '24', 'Massachusetts': '25', 'Michigan': '26',
    'Minnesota': '27', 'Mississippi': '28', 'Missouri': '29', 'Montana': '30',
    'Nebraska': '31', 'Nevada': '32', 'New Hampshire': '33', 'New Jersey': '34',
    'New Mexico': '35', 'New York': '36', 'North Carolina': '37', 'North Dakota': '38',
    'Ohio': '39', 'Oklahoma': '40', 'Oregon': '41', 'Pennsylvania': '42',
    'Rhode Island': '44', 'South Carolina': '45', 'South Dakota': '46',
    'Tennessee': '47', 'Texas': '48', 'Utah': '49', 'Vermont': '50',
    'Virginia': '51', 'Washington': '53', 'West Virginia': '54', 'Wisconsin': '55',
    'Wyoming': '56'
}

## 1. Dissolve state features into single contiguous feature

In [ ]:
input_states = os.path.join(data_dir, "US_Contig_State_Boundaries", "US_Contig_State_Boundaries.shp")
us_contig_boundary_dir = os.path.join(data_dir, "US_Contiguous_Boundary")
contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary.shp")
os.makedirs(us_contig_boundary_dir, exist_ok=True)

print("Dissolving 48 state boundaries into a single contiguous feature...")

arcpy.management.Dissolve(
    in_features=input_states,
    out_feature_class=contiguous_us
)

print(f"Success! Master boundary saved to: {contiguous_us}")

Dissolving 48 state boundaries into a single contiguous feature...
Success! Master boundary saved to: D:\Projects\suitable-data-center\data\US_Contiguous_Boundary\US_Contiguous_Boundary.shp


## 2. Reproject Boundary to EPSG:5070 (NAD83 / Conus Albers)

In [ ]:
projected_contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary_5070.shp")

print("Projecting boundary to EPSG:5070...")

arcpy.management.Project(
    in_dataset=contiguous_us,
    out_dataset=projected_contiguous_us,
    out_coor_system=conus_albers_sr
)

print(f"Success! Projected boundary saved to: {projected_contiguous_us}")

Projecting boundary to EPSG:5070...
Success! Projected boundary saved to: D:\Projects\suitable-data-center\data\US_Contiguous_Boundary\US_Contiguous_Boundary_5070.shp


## 3. Slope Raster
Clip and standardize slope raster

In [ ]:
raw_slope_tif = os.path.join(data_dir, "slope_1KMmd_GMTEDmd.tif")
slope_processing_dir = os.path.join(data_dir, "Slope_Processing")
os.makedirs(slope_processing_dir, exist_ok=True)

clipped_slope_tif = os.path.join(slope_processing_dir, "Slope_Clipped_5070.tif")
standardized_slope_tif = os.path.join(slope_processing_dir, "Slope_Standardized.tif")

print("Extracting slope raster using the contiguous US boundary mask...")

with arcpy.EnvManager(outputCoordinateSystem=conus_albers_sr, snapRaster=raw_slope_tif):
    clipped_slope = ExtractByMask(
        in_raster=raw_slope_tif,
        in_mask_data=projected_contiguous_us
    )
    clipped_slope.save(clipped_slope_tif)

print("Excluding slopes > 10 degrees and scaling remaining areas from 0.0 to 1.0...")

# Map Algebra logic:
# 1. 'clipped_slope <= 10' acts as a conditional filter. Anything > 10 becomes NoData.
# 2. '1.0 - (clipped_slope / 10.0)' inverts and normalizes the scale.
#    A slope of 0 degrees becomes 1.0 (highly suitable).
#    A slope of 10 degrees becomes 0.0 (least suitable).
standardized_slope = Con(clipped_slope <= 10, 1.0 - (clipped_slope / 10.0))
standardized_slope.save(standardized_slope_tif)

print(f"Success! Standardized slope raster saved to: {standardized_slope_tif}")

## 4. Aggregate Slope Suitability to Counties

In [ ]:
print("Aggregating standardized slope suitability to counties...")

slope_zonal_table = os.path.join(slope_processing_dir, "Slope_Zonal_Mean.dbf")

ZonalStatisticsAsTable(
    in_zone_data=master_counties,
    zone_field="GEOID",
    in_value_raster=standardized_slope_tif,
    out_table=slope_zonal_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

# Read zonal mean values into dictionary
slope_dict = {}
with arcpy.da.SearchCursor(slope_zonal_table, ["GEOID", "MEAN"]) as cursor:
    for geoid, mean_val in cursor:
        if mean_val is not None:
            slope_dict[str(geoid).zfill(5)] = float(mean_val)

# Add fields if needed
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "SlopeScr" not in existing_fields:
    arcpy.management.AddField(master_counties, "SlopeScr", "DOUBLE", field_is_nullable=True)

# Update county master shapefile
with arcpy.da.UpdateCursor(master_counties, ["GEOID", "SlopeScr"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        if geoid in slope_dict:
            row[1] = slope_dict[geoid]
            cursor.updateRow(row)

print("Success! Slope county scores added to Counties_Suitability_Master.")

## 5. Proximity to Water Raster
Generate and standardize distance to water surface

Note to future selves: a limitation of this is that it does not account for underground water sources nor the volume of water, so it's flawed

In [ ]:
lakes_shp = os.path.join(data_dir, "rivers_and_lakes_shapefile", "NA_Lakes_and_Rivers", "data", "lakes_p", "northamerica_lakes_cec_2023.shp")
rivers_shp = os.path.join(data_dir, "rivers_and_lakes_shapefile", "NA_Lakes_and_Rivers", "data", "rivers_l", "northamerica_rivers_cec_2023.shp")

water_processing_dir = os.path.join(data_dir, "Water_Processing")
os.makedirs(water_processing_dir, exist_ok=True)

dist_water_tif = os.path.join(water_processing_dir, "Distance_to_Water_5070.tif")
standardized_water_tif = os.path.join(water_processing_dir, "Distance_to_Water_Standardized.tif")

print("Calculating distance to lakes and rivers within the US boundary...")

with arcpy.EnvManager(extent=projected_contiguous_us, mask=projected_contiguous_us, outputCoordinateSystem=conus_albers_sr, cellSize=1000):    
    dist_lakes = EucDistance(lakes_shp)    
    dist_rivers = EucDistance(rivers_shp)
    
    min_dist_water = CellStatistics([dist_lakes, dist_rivers], "MINIMUM", "DATA")
    min_dist_water.save(dist_water_tif)

print("Standardizing water distance to a 0.0 to 1.0 suitability scale...")

max_dist = float(arcpy.management.GetRasterProperties(min_dist_water, "MAXIMUM").getOutput(0))

# Invert and normalize the scale: 
# A distance of 0 becomes 1.0 (highly suitable), the maximum distance becomes 0.0
standardized_water = 1.0 - (min_dist_water / max_dist)
standardized_water.save(standardized_water_tif)

print(f"Success! Standardized water raster saved to: {standardized_water_tif}")

## 6. Aggregate Water Suitability to Counties

In [ ]:
print("Aggregating standardized water suitability to counties...")

water_zonal_table = os.path.join(water_processing_dir, "Water_Zonal_Mean.dbf")

ZonalStatisticsAsTable(
    in_zone_data=master_counties,
    zone_field="GEOID",
    in_value_raster=standardized_water_tif,
    out_table=water_zonal_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

# Read zonal mean values into dictionary
water_dict = {}
with arcpy.da.SearchCursor(water_zonal_table, ["GEOID", "MEAN"]) as cursor:
    for geoid, mean_val in cursor:
        if mean_val is not None:
            water_dict[str(geoid).zfill(5)] = float(mean_val)

# Add fields if needed
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "Wtr_Score" not in existing_fields:
    arcpy.management.AddField(master_counties, "Wtr_Score", "DOUBLE", field_is_nullable=True)

# Update county master shapefile
with arcpy.da.UpdateCursor(master_counties, ["GEOID", "Wtr_Score"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        if geoid in water_dict:
            row[1] = water_dict[geoid]
            cursor.updateRow(row)

print("Success! Water county scores added to Counties_Suitability_Master.")

## 7. Climate / Temperature Processing
Join aggregated county temperatures and standardize to a suitability raster.

In [ ]:
print("Step 1: Processing NOAA temperatures using Pandas...")
avg_temp_csv = os.path.join(data_dir, "avg_temp_counties.csv")
df = pd.read_csv(avg_temp_csv, skiprows=3)

df['GEOID'] = df['State'].map(state_fips) + df['ID'].str.split('-').str[1]
temp_dict = dict(zip(df['GEOID'], df['Value']))

min_temp = df['Value'].min()
max_temp = df['Value'].max()

print("Step 2: Applying the Virginia Independent City Patch...")
# This maps the weird independent cities to their surrounding county GEOID
va_patch = {
    "51510": "51059", "51600": "51059", "51610": "51059", # Fairfax County surrounds
    "51540": "51003", # Charlottesville -> Albemarle
    "51760": "51087", # Richmond City -> Henrico
    "51770": "51161", # Roanoke City -> Roanoke County
    "51810": "51163", # Virginia Beach -> Rockbridge (proxy)
    "11001": "24031"  # Washington DC -> Montgomery County, MD
}

for city_geoid, county_geoid in va_patch.items():
    if county_geoid in temp_dict:
        temp_dict[city_geoid] = temp_dict[county_geoid]

print("Step 3: Injecting data and standardizing scores in the attribute table...")
arcpy.management.AddField(master_counties, "Temp_Raw", "DOUBLE", field_is_nullable=True)
arcpy.management.AddField(master_counties, "Temp_Score", "DOUBLE", field_is_nullable=True)

with arcpy.da.UpdateCursor(master_counties, ["GEOID", "Temp_Raw", "Temp_Score"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        
        if geoid in temp_dict:
            raw_val = float(temp_dict[geoid])
            row[1] = raw_val
            row[2] = 1.0 - ((raw_val - min_temp) / (max_temp - min_temp))
            cursor.updateRow(row)
        else:
            continue

print("Success! Temperature values and standardized scores added to Counties_Suitability_Master.")

## 8. Electricity Prices

In [ ]:
print("Step 1: Loading State Electricity Prices...")
elec_csv = os.path.join(data_dir, "electric_price_by_state.csv")

df_elec = pd.read_csv(elec_csv, skiprows=2)
price_column = "Industrial"

df_elec['State_FIPS'] = df_elec['State'].map(state_fips)

df_elec = df_elec.dropna(subset=[price_column])
elec_dict = dict(zip(df_elec['State_FIPS'], df_elec[price_column]))

min_price = df_elec[price_column].min()
max_price = df_elec[price_column].max()

print("Step 2: Injecting data into Counties_Suitability_Master...")
arcpy.management.AddField(master_counties, "Elec_Raw", "DOUBLE", field_is_nullable=True)
arcpy.management.AddField(master_counties, "Elec_Score", "DOUBLE", field_is_nullable=True)

with arcpy.da.UpdateCursor(master_counties, ["GEOID", "Elec_Raw", "Elec_Score"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        state_code = geoid[:2]

        if state_code in elec_dict:
            raw_val = float(elec_dict[state_code])
            row[1] = raw_val
            row[2] = 1.0 - ((raw_val - min_price) / (max_price - min_price))
            
            cursor.updateRow(row)
        else:
            continue

print("Success! Industrial electricity prices and scores added to Counties_Suitability_Master")

## 9. Access to Electricity

In [ ]:
print("Processing electricity access...")

## 1. Define input and output paths
projected_contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary_5070.shp")

transmission_dir = os.path.join(data_dir, "US_Electric_Power_Transmission_Lines")
electricity_processing_dir = os.path.join(data_dir, "Electricity_Processing")
os.makedirs(electricity_processing_dir, exist_ok=True)

transmission_lines = os.path.join(transmission_dir, "Electric_Power_Transmission_Lines_A.shp")
transmission_5070 = os.path.join(electricity_processing_dir, "TransLn_5070.shp")
clipped_lines = os.path.join(electricity_processing_dir, "TransLn_Clip.shp")
operational_lines = os.path.join(electricity_processing_dir, "TransLn_On.shp")

distance_raster = os.path.join(electricity_processing_dir, "ElecDist.tif")
distance_score_raster = os.path.join(electricity_processing_dir, "ElecDistSc.tif")
electricity_score_raster = os.path.join(electricity_processing_dir, "ElecAccSc.tif")
electricity_zonal_table = os.path.join(electricity_processing_dir, "ElecZon.dbf")

## 2. Project transmission lines to the common coordinate system
arcpy.management.Project(
    in_dataset=transmission_lines,
    out_dataset=transmission_5070,
    out_coor_system=conus_albers_sr
)

## 3. Clip to the AOI to remove lines in Alaska, Hawaii, Guam, etc.
arcpy.analysis.Clip(
    in_features=transmission_5070,
    clip_features=projected_contiguous_us,
    out_feature_class=clipped_lines,
)

## 4. Select only lines that are currently in service
arcpy.analysis.Select(
    in_features=clipped_lines,
    out_feature_class=operational_lines,
    where_clause="STATUS = 'IN SERVICE'"
)

## 5. Add a voltage suitability field and score each line
existing_fields = [f.name for f in arcpy.ListFields(operational_lines)]
if "VoltScr" not in existing_fields:
    arcpy.management.AddField(operational_lines, "VoltScr", "DOUBLE")

with arcpy.da.UpdateCursor(operational_lines, ["VOLTAGE", "VoltScr"]) as cursor:
    for row in cursor:
        v = row[0]

        # Assign higher scores to higher-voltage lines
        if v is None or v < 0:
            row[1] = 0.0
        elif v >= 500:
            row[1] = 1.0
        elif v >= 345:
            row[1] = 0.8
        elif v >= 230:
            row[1] = 0.6
        elif v >= 115:
            row[1] = 0.4
        else:
            row[1] = 0.2

        cursor.updateRow(row)

## 6. Create Euclidean distance raster from in-service transmission lines
dist_raster = EucDistance(
    in_source_data=operational_lines,
    cell_size=1000
)
dist_raster.save(distance_raster)

## 7. Standardize distance so that closer lines receive higher suitability
distance_score = Con(
    Raster(distance_raster) <= 20000,
    1 - (Raster(distance_raster) / 20000.0),
    0
)
distance_score.save(distance_score_raster)

## 8. Convert voltage scores from lines to raster
volt_raster = os.path.join(electricity_processing_dir, "VoltScr.tif")

arcpy.conversion.PolylineToRaster(
    in_features=operational_lines,
    value_field="VoltScr",
    out_rasterdataset=volt_raster,
    cell_assignment="MAXIMUM_LENGTH",
    priority_field="VoltScr",
    cellsize=1000
)

## Replace NoData with 0 before combining rasters
voltage_score = Con(IsNull(Raster(volt_raster)), 0, Raster(volt_raster))

## 9. Combine distance and voltage into one electricity access suitability raster
combined_score = (Raster(distance_score_raster) * 0.7) + (voltage_score * 0.3)
combined_score.save(electricity_score_raster)

## 10. Aggregate raster values to the county level using zonal statistics
ZonalStatisticsAsTable(
    in_zone_data=master_counties,
    zone_field="GEOID",
    in_value_raster=electricity_score_raster,
    out_table=electricity_zonal_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

## Read county-level mean electricity access scores into a dictionary
elec_access_dict = {}
with arcpy.da.SearchCursor(electricity_zonal_table, ["GEOID", "MEAN"]) as cursor:
    for geoid, mean_val in cursor:
        if mean_val is not None:
            elec_access_dict[str(geoid).zfill(5)] = float(mean_val)

## Add the electricity access field to the county master layer if needed
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "ElecAcc" not in existing_fields:
    arcpy.management.AddField(master_counties, "ElecAcc", "DOUBLE")

## Write county-level electricity access scores into the master county shapefile
with arcpy.da.UpdateCursor(master_counties, ["GEOID", "ElecAcc"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        if geoid in elec_access_dict:
            row[1] = elec_access_dict[geoid]
            cursor.updateRow(row)

print("Success! Electricity access added to county master layer.")

## 10. Broadband Avaliability

Uses the mean advertised speeds per county

In [ ]:
print("Processing broadband availability...")

## 1. Define input and output paths
broadband_dir = os.path.join(data_dir, "network_availability")
broadband_processing_dir = os.path.join(data_dir, "Broadband_Processing")
os.makedirs(broadband_processing_dir, exist_ok=True)

broadband_combined_csv = os.path.join(broadband_processing_dir, "Broadband_AllStates.csv")

## 2. Read and combine all broadband CSV files from state subfolders
dfs = []

for state_folder in os.listdir(broadband_dir):
    state_folder_path = os.path.join(broadband_dir, state_folder)

    if os.path.isdir(state_folder_path):
        for file in os.listdir(state_folder_path):
            if file.lower().endswith(".csv"):
                file_path = os.path.join(state_folder_path, file)
                df = pd.read_csv(file_path)
                dfs.append(df)

broadband_df = pd.concat(dfs, ignore_index=True)
broadband_df.to_csv(broadband_combined_csv, index=False)

## 3. Standardize block GEOID and derive county GEOID
broadband_df["block_geoid"] = broadband_df["block_geoid"].astype(str).str.replace(".0", "", regex=False)
broadband_df["CountyGEOID"] = broadband_df["block_geoid"].str.zfill(15).str[:5]

## 4. Keep only rows with usable speed values
broadband_df["max_advertised_download_speed"] = pd.to_numeric(
    broadband_df["max_advertised_download_speed"], errors="coerce"
)
broadband_df["max_advertised_upload_speed"] = pd.to_numeric(
    broadband_df["max_advertised_upload_speed"], errors="coerce"
)

broadband_df = broadband_df.dropna(
    subset=["CountyGEOID", "max_advertised_download_speed", "max_advertised_upload_speed"]
)

## 5. Aggregate broadband speeds to the county level
county_broadband = broadband_df.groupby("CountyGEOID", as_index=False).agg(
    AvgDown=("max_advertised_download_speed", "mean"),
    AvgUp=("max_advertised_upload_speed", "mean")
)

## 6. Create a combined broadband suitability input
county_broadband["BroadRaw"] = (
    0.7 * county_broadband["AvgDown"] +
    0.3 * county_broadband["AvgUp"]
)

## 7. Standardize broadband values to a 0-1 suitability scale
min_val = county_broadband["BroadRaw"].min()
max_val = county_broadband["BroadRaw"].max()

if max_val != min_val:
    county_broadband["BroadSc"] = (
        county_broadband["BroadRaw"] - min_val
    ) / (max_val - min_val)
else:
    county_broadband["BroadSc"] = 0.0

## 8. Convert county broadband scores to a dictionary
broadband_dict = dict(zip(county_broadband["CountyGEOID"], county_broadband["BroadSc"]))

## 9. Add broadband field to the county master layer if needed
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "BroadSc" not in existing_fields:
    arcpy.management.AddField(master_counties, "BroadSc", "DOUBLE")

## 10. Write broadband scores into the county master shapefile
with arcpy.da.UpdateCursor(master_counties, ["GEOID", "BroadSc"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        if geoid in broadband_dict:
            row[1] = float(broadband_dict[geoid])
            cursor.updateRow(row)

print("Success! Broadband availability added to county master layer.")

## 11. National Risk Index
Maps the risk index to each county

In [ ]:
print("Step 1: Reading National Risk Index data into memory...")
nri_shp = os.path.join(data_dir, "National_Risk_Index_Counties", "NRI_Counties_Prod.shp")

nri_id_field = "STCOFIPS"
nri_score_field = "RISK_SCORE"

risk_dict = {}
risk_values = []

with arcpy.da.SearchCursor(nri_shp, [nri_id_field, nri_score_field]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        raw_score = row[1]

        if raw_score is not None and isinstance(raw_score, (int, float)):
            risk_dict[geoid] = float(raw_score)
            risk_values.append(float(raw_score))

min_risk = min(risk_values)
max_risk = max(risk_values)

print(f" -> Found {len(risk_values)} valid risk records")

print("Step 2: Injecting data into Counties_Suitability_Master")
arcpy.management.AddField(master_counties, "Risk_Raw", "DOUBLE", field_is_nullable=True)
arcpy.management.AddField(master_counties, "Risk_Score", "DOUBLE", field_is_nullable=True)

with arcpy.da.UpdateCursor(master_counties, ["GEOID", "Risk_Raw", "Risk_Score"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        if geoid in risk_dict:
            raw_val = risk_dict[geoid]
            row[1] = raw_val
            row[2] = 1.0 - ((raw_val - min_risk) / (max_risk - min_risk))
            cursor.updateRow(row)
        else:
            continue

print("Success! Disaster risk scores added to Countries_Suitability_Master")

## 12. Transportation Infrastructure

In [ ]:
print("Processing transportation infrastructure...")

# 1. Define input and output paths
transportation_dir = os.path.join(data_dir, "NTAD_Freight_Analysis_Framework_Network_Links")
trans_processing_dir = os.path.join(data_dir, "Transportation_Processing")
os.makedirs(trans_processing_dir, exist_ok=True)

faf5_lines = os.path.join(transportation_dir, "Freight_Analysis_Framework_(FAF5)_Network_Links.shp") 

trans_5070 = os.path.join(trans_processing_dir, "Trans_5070.shp")
clipped_trans = os.path.join(trans_processing_dir, "Trans_Clip.shp")

distance_raster_trans = os.path.join(trans_processing_dir, "TransDist.tif")
distance_score_trans = os.path.join(trans_processing_dir, "TransDistSc.tif")
trans_score_raster = os.path.join(trans_processing_dir, "TransAccSc.tif")
trans_zonal_table = os.path.join(trans_processing_dir, "TransZon.dbf")

# 2. Project transportation lines to EPSG:5070 (NAD83 / Conus Albers)
arcpy.management.Project(
    in_dataset=faf5_lines,
    out_dataset=trans_5070,
    out_coor_system=conus_albers_sr
)

# 3. Clip to the Contiguous US Boundary
arcpy.analysis.Clip(
    in_features=trans_5070,
    clip_features=projected_contiguous_us,
    out_feature_class=clipped_trans,
)

# 4. Add a suitability field for road class/capacity and score each line
# FAF5 typically uses F_Class (1=Interstate, 2=Freeway, 3=Principal Arterial)
arcpy.management.AddField(clipped_trans, "RoadScr", "DOUBLE")

with arcpy.da.UpdateCursor(clipped_trans, ["F_Class", "RoadScr"]) as cursor:
    for row in cursor:
        fclass = row[0]

        # Assign higher scores to high-capacity logistics routes like interstates
        if fclass is None:
            row[1] = 0.0
        elif fclass == 1:       # Interstate
            row[1] = 1.0
        elif fclass == 2:       # Freeway / Expressway
            row[1] = 0.8
        elif fclass == 3:       # Principal Arterial
            row[1] = 0.6
        elif fclass == 4:       # Minor Arterial
            row[1] = 0.4
        else:                   # Local/Collector roads
            row[1] = 0.2

        cursor.updateRow(row)

# 5. Create Euclidean distance raster from the network links
print(" -> Calculating Euclidean distance...")
dist_raster_trans = EucDistance(
    in_source_data=clipped_trans,
    cell_size=1000
)
dist_raster_trans.save(distance_raster_trans)

# 6. Standardize distance (e.g., within 20km is ideal for logistics)
# 1.0 is exactly on the network, degrading to 0.0 at 20km away
print(" -> Standardizing distance scores...")
dist_score_trans = Con(
    Raster(distance_raster_trans) <= 20000,
    1 - (Raster(distance_raster_trans) / 20000.0),
    0
)
dist_score_trans.save(distance_score_trans)

# 7. Convert the road class scores into a raster
road_class_raster = os.path.join(trans_processing_dir, "RoadScr.tif")

arcpy.conversion.PolylineToRaster(
    in_features=clipped_trans,
    value_field="RoadScr",
    out_rasterdataset=road_class_raster,
    cell_assignment="MAXIMUM_LENGTH",
    priority_field="RoadScr",
    cellsize=1000
)

# Replace NoData with 0
road_score_clean = Con(IsNull(Raster(road_class_raster)), 0, Raster(road_class_raster))

# 8. Combine distance proximity and road capacity into one composite raster
print(" -> Combining distance and capacity scores...")
combined_trans_score = (Raster(distance_score_trans) * 0.7) + (road_score_clean * 0.3)
combined_trans_score.save(trans_score_raster)

# 9. Aggregate raster values to the county level using zonal statistics
print(" -> Running Zonal Statistics...")
ZonalStatisticsAsTable(
    in_zone_data=master_counties,
    zone_field="GEOID",
    in_value_raster=trans_score_raster,
    out_table=trans_zonal_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

# 10. Read county-level mean transportation scores into a dictionary
trans_access_dict = {}
with arcpy.da.SearchCursor(trans_zonal_table, ["GEOID", "MEAN"]) as cursor:
    for geoid, mean_val in cursor:
        if mean_val is not None:
            trans_access_dict[str(geoid).zfill(5)] = float(mean_val)

# Add the transportation access field to the county master layer
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "TransAcc" not in existing_fields:
    arcpy.management.AddField(master_counties, "TransAcc", "DOUBLE", field_is_nullable="NULLABLE")

# Write county-level transportation scores into the master county shapefile
print(" -> Writing to master county layer...")
with arcpy.da.UpdateCursor(master_counties, ["GEOID", "TransAcc"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        if geoid in trans_access_dict:
            row[1] = trans_access_dict[geoid]
            cursor.updateRow(row)
        else:
            continue

print("Success! Transportation access added to county master layer.")

Processing transportation infrastructure...
 -> Calculating Euclidean distance...
 -> Standardizing distance scores...
 -> Combining distance and capacity scores...
 -> Running Zonal Statistics...
 -> Writing to master county layer...
Success! Transportation access added to county master layer.


## 13. Land Cover
This one takes a loooonnnggg time

In [ ]:
nlcd_raster = os.path.join(data_dir, "Annual_NLCD_LndCov_2024_CU_C1V1", "Annual_NLCD_LndCov_2024_CU_C1V1.tif")
nlcd_table = os.path.join(data_dir, "NLCD_Tabulated_Areas.dbf")

print("Step 1: Calculating exact area of each land cover type per county...")
TabulateArea(
    in_zone_data=master_counties,
    zone_field="GEOID",
    in_class_data=nlcd_raster,
    class_field="Value",
    out_table=nlcd_table,
    processing_cell_size=30 # 30m resolution
)

print("Step 2: Applying suitability weights to land categories...")
nlcd_weights = {
    '11': 0.0, '12': 0.0,            # Water, Ice
    
    # Developed land (low to high intensity):
    '21': 0.7, '22': 0.6, '23': 0.3, '24': 0.1,
    '31': 1.0,                       # Barren land
    '41': 0.4, '42': 0.4, '43': 0.4, # Forest
    '51': 0.8, '52': 0.8,            # Shrubland
    
    # Grassland:
    '71': 0.8, '72': 0.8, '73': 0.8, '74': 0.8,
    '81': 0.5, '82': 0.5,            # Agriculture
    '90': 0.0, '95': 0.0             # Wetlands
}

fields = [f.name for f in arcpy.ListFields(nlcd_table)]
print("NLCD table fields:", fields)

value_fields = [f for f in fields if f.startswith("VALUE_")]
land_scores = {}

zone_field_name = "GEOID"
if zone_field_name not in fields:
    raise ValueError(f"{zone_field_name} not found in {nlcd_table}. Available fields: {fields}")

with arcpy.da.SearchCursor(nlcd_table, [zone_field_name] + value_fields) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
        total_area = 0
        weighted_area = 0

        for i, field in enumerate(value_fields):
            area = row[i+1]
            if area and area > 0:
                code = field.split("_")[1]
                weight = nlcd_weights.get(code, 0.0)

                total_area += area
                weighted_area += (area * weight)
        
        if total_area > 0:
            land_scores[geoid] = weighted_area / total_area

print("Step 3: Injecting the data into Counties_Suitability_Master...")
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "Land_Score" not in existing_fields:
    arcpy.management.AddField(master_counties, "Land_Score", "DOUBLE", field_is_nullable=True)

with arcpy.da.UpdateCursor(master_counties, ["GEOID", "Land_Score"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5)
    
        if geoid in land_scores:
            row[1] = land_scores[geoid]
            cursor.updateRow(row)
        else:
            continue

print("Success! Land Cover scores added to Countries_Suitability_Master")

### Establish Criteria Weights

In [ ]:
# Joels proposed weights:
# weights = {
#     'Elec_Score': 0.2,  # 20% (Cost of Electricity)
#     'ElecAcc': 0.15,    # 15% (Access to Electricity)
#     'BroadSc': 0.15,    # 15%
#     'Risk_Score': 0.15, # 15%
#     'Temp_Score': 0.1,  # 10%
#     'Wtr_Score': 0.1,   # 10%
#     'TransAcc': 0.05,   # 5%
#     'SlopeScr': 0.05,   # 5%
#     'Land_Score': 0.05, # 5%
# }
weights = {
    "BroadSc": 0.30,
    "ElecAcc": 0.25,
    "Elec_Score": 0.15,
    "TransAcc": 0.10,
    "Temp_Score": 0.05,
    "Wtr_Score": 0.05,
    "Risk_Score": 0.05,
    "Land_Score": 0.03,
    "SlopeScr": 0.02,
}
criteria_fields = list(weights.keys())

total_weight = sum(weights.values())
if not math.isclose(total_weight, 1.0):
    print(f"You suck at math, the weights add to {total_weight}")
else:
    print("Success!")

Success!


In [ ]:
existing_fields = [f.name for f in arcpy.ListFields(master_counties)]
if "FinalScore" not in existing_fields:
    arcpy.management.AddField(master_counties, "FinalScore", "DOUBLE", field_is_nullable="NULLABLE")

missing_fields = [f for f in criteria_fields if f not in existing_fields]
if len(missing_fields):
    raise SystemError(f"Master Counties Shapefile is missing these fields: {missing_fields}")

needed_fields = criteria_fields + ["FinalScore"]

success_count = 0
missing_data_count = 0

with arcpy.da.UpdateCursor(master_counties, needed_fields) as cursor:
    for row in cursor:
        row_values = row[:-1]
        
        if None in row_values:
            row[-1] = None
            missing_data_count += 1
            cursor.updateRow(row)
            continue
            
        final_composite_score = 0.0
        
        for index, field_name in enumerate(criteria_fields):
            score = float(row_values[index])
            weight = weights[field_name]
            final_composite_score += (score * weight)
            
        row[-1] = final_composite_score
        cursor.updateRow(row)
        success_count += 1

print(f"Model Complete!")
print(f" -> Successfully scored: {success_count} counties.")
print(f" -> Skipped (missing data): {missing_data_count} counties.")

In [ ]:
fields = ["NAMELSAD", "State_Name", "FinalScore"]

results = []

with arcpy.da.SearchCursor(master_counties, fields) as cursor:
    for row in cursor:
        name = row[0]
        state = row[1]
        score = row[2]
        
        if score is not None:
            results.append((name, state, score))

results.sort(key=lambda x: x[2], reverse=True)

print("Top 5 Data Center Candidate Locations")

for i in range(5):
    name, state, score = results[i]
    print(f" {i+1}. {name}, {state} | Suitability Score: {score:.4f}")


top_states = {}

for i in range(100):
    name, state, score = results[i]
    if state in top_states:
        top_states[state] += 1
    else:
        top_states[state] = 1

states_results = [(s, c) for s, c in top_states.items()]
states_results.sort(key=lambda x: x[1], reverse=True)

print("Of the Top 100 Counties...")
for state, count in states_results:
    word = "is" if count == 1 else "are"
    print(f" -> {count} {word} in {state}")